In [ ]:
import sys
import heapq # Added for Prim's and Dijkstra's

# Set a higher recursion limit for algorithms like Quick Sort
sys.setrecursionlimit(2000)

# =================================================================================================
# 8. Kruskal's and Prim's Algorithms (Minimum Spanning Tree)
# =================================================================================================
Both find a Minimum Spanning Tree (MST).
Kruskal's: Edge-based, sorts all edges.
Prim's: Vertex-based, grows a tree from a start node.

In [3]:
# --- Kruskal's Algorithm ---
# Helper functions for DSU (Disjoint Set Union)
def kruskal_find(parent_map, i):
    if parent_map[i] == i:
        return i
    return kruskal_find(parent_map, parent_map[i])

def kruskal_union(parent_map, rank, x, y):
    xroot = kruskal_find(parent_map, x)
    yroot = kruskal_find(parent_map, y)
    if rank[xroot] < rank[yroot]:
        parent_map[xroot] = yroot
    elif rank[xroot] > rank[yroot]:
        parent_map[yroot] = xroot
    else:
        parent_map[yroot] = xroot
        rank[xroot] += 1

def kruskal_mst(edges_list, vertices):
    """
    Finds MST using Kruskal's algorithm.
    edges_list: List of (weight, v1, v2)
    vertices: List of vertex labels
    """
    result_mst = []
    edges_list.sort() # Sort edges by weight
    
    parent_map = {v: v for v in vertices}
    rank = {v: 0 for v in vertices}
    
    edge_count = 0
    i = 0
    while edge_count < len(vertices) - 1 and i < len(edges_list):
        weight, v1, v2 = edges_list[i]
        i += 1
        x = kruskal_find(parent_map, v1)
        y = kruskal_find(parent_map, v2)
        
        if x != y:
            edge_count += 1
            result_mst.append((v1, v2, weight))
            kruskal_union(parent_map, rank, x, y)
            
    return result_mst


# =================================================================================================
# 9. Dijkstra's Algorithm (Single Source Shortest Path)
# =================================================================================================
Finds the shortest path from a source vertex to all other vertices.
Requires non-negative edge weights.

In [4]:
def dijkstra(graph, start_node):
    distances = {node: float('infinity') for node in graph}
    distances[start_node] = 0
    
    # Priority queue stores (distance, node)
    priority_queue = [(0, start_node)]
    
    while priority_queue:
        current_dist, current_node = heapq.heappop(priority_queue)
        
        # If we've found a shorter path already, skip
        if current_dist > distances[current_node]:
            continue
            
        for weight, neighbor in graph[current_node]:
            distance = current_dist + weight
            # If we found a shorter path to the neighbor
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))
                
    return distances

def test_dijkstra():
    print("--- 9. Dijkstra's Algorithm ---")
    # Using the same graph as the MST example
    graph_adj = {
        'A': [(2, 'B'), (5, 'C')],
        'B': [(2, 'A'), (4, 'C'), (3, 'D')],
        'C': [(5, 'A'), (4, 'B'), (1, 'D')],
        'D': [(3, 'B'), (1, 'C')]
    }
    start_node = 'A'
    
    shortest_paths = dijkstra(graph_adj, start_node)
    print(f"Graph: {graph_adj}")
    print(f"Shortest paths from source '{start_node}': {shortest_paths}")
    print("-" * 31 + "\n")

In [5]:
# --- Prim's Algorithm ---
def prim_mst(graph_adj, start_node):
    """
    Finds MST using Prim's algorithm.
    graph_adj: Adjacency list {vertex: [(weight, neighbor), ...]}
    """
    visited = {start_node}
    priority_queue = [(weight, start_node, neighbor) for weight, neighbor in graph_adj[start_node]]
    heapq.heapify(priority_queue)
    
    mst = []
    mst_cost = 0
    
    while priority_queue:
        weight, v1, v2 = heapq.heappop(priority_queue)
        if v2 not in visited:
            visited.add(v2)
            mst.append((v1, v2, weight))
            mst_cost += weight
            for w_new, neighbor_new in graph_adj[v2]:
                if neighbor_new not in visited:
                    heapq.heappush(priority_queue, (w_new, v2, neighbor_new))
                    
    return mst, mst_cost

def test_mst_algorithms():
    print("--- 8. Kruskal's and Prim's MST Algorithms ---")
    vertices = ['A', 'B', 'C', 'D']
    # (weight, v1, v2)
    edges_list = [
        (2, 'A', 'B'), (5, 'A', 'C'), (4, 'B', 'C'),
        (3, 'B', 'D'), (1, 'C', 'D')
    ]
    # {vertex: [(weight, neighbor), ...]}
    graph_adj = {
        'A': [(2, 'B'), (5, 'C')],
        'B': [(2, 'A'), (4, 'C'), (3, 'D')],
        'C': [(5, 'A'), (4, 'B'), (1, 'D')],
        'D': [(3, 'B'), (1, 'C')]
    }
    
    print(f"Graph Vertices: {vertices}")
    print(f"Graph Edges: {edges_list}")

    kruskal_result = kruskal_mst(edges_list, vertices)
    print(f"\nKruskal's MST Edges (v1, v2, weight): {kruskal_result}")
    print(f"Kruskal's Total Cost: {sum(w for _, _, w in kruskal_result)}")
    
    prim_result, prim_cost = prim_mst(graph_adj, 'A')
    print(f"\nPrim's MST Edges (v1, v2, weight): {prim_result}")
    print(f"Prim's Total Cost: {prim_cost}")
    print("-" * 40 + "\n")

# =================================================================================================
# 10. Sudoku Solver (Backtracking)
# =================================================================================================
Solves a 9x9 Sudoku puzzle using backtracking.

In [6]:
def print_sudoku_grid(grid):
    for row in grid:
        print(row)

def is_safe_sudoku(grid, row, col, num):
    # Check row
    if num in grid[row]:
        return False
        
    # Check column
    if num in [grid[i][col] for i in range(9)]:
        return False
        
    # Check 3x3 box
    start_row, start_col = 3 * (row // 3), 3 * (col // 3)
    for i in range(start_row, start_row + 3):
        for j in range(start_col, start_col + 3):
            if grid[i][j] == num:
                return False
                
    return True

def solve_sudoku(grid):
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:  # Found an empty cell
                for num in range(1, 10): # Try numbers 1-9
                    if is_safe_sudoku(grid, r, c, num):
                        grid[r][c] = num # Place number
                        if solve_sudoku(grid): # Recurse
                            return True # Solution found!
                        
                        grid[r][c] = 0 # Backtrack
                return False # No number (1-9) worked
                
    return True # No empty cells, puzzle is solved

def test_sudoku_solver():
    print("--- 10. Sudoku Solver (Backtracking) ---")
    puzzle = [
        [5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9]
    ]
    print("Puzzle to solve:")
    print_sudoku_grid(puzzle)
    
    if solve_sudoku(puzzle):
        print("\nSudoku solved!")
        print_sudoku_grid(puzzle)
    else:
        print("\nNo solution exists.")
    print("-" * 40 + "\n")

# =================================================================================================
# 11. Floyd-Warshall Algorithm (All-Pairs Shortest Paths)
# =================================================================================================
Finds the shortest path between all pairs of vertices.

In [7]:
def floyd_warshall(graph_matrix):
    V = len(graph_matrix)
    # Initialize the solution matrix same as input graph
    dist = [row[:] for row in graph_matrix]
    
    # The core algorithm in 3 nested loops
    for k in range(V):      # k is the intermediate vertex
        for i in range(V):  # i is the source vertex
            for j in range(V): # j is the destination vertex
                
                # Is the path from i -> k -> j shorter?
                dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
                
    return dist

def test_floyd_warshall():
    print("--- 11. Floyd-Warshall Algorithm ---")
    V = 3
    INF = float('inf')
    # Adjacency Matrix for graph A=0, B=1, C=2
    graph_matrix = [
        [0, 3, 8],      # Costs from A (A->A=0, A->B=3, A->C=8)
        [INF, 0, 2],    # Costs from B (B->A=inf, B->B=0, B->C=2)
        [INF, INF, 0]   # Costs from C (C->A=inf, C->B=inf, C->C=0)
    ]
    print("Initial Cost Matrix (A=0, B=1, C=2):")
    for row in graph_matrix:
        print(row)
        
    shortest_paths = floyd_warshall(graph_matrix)

    print("\nAll-Pairs Shortest Path Matrix:")
    for row in shortest_paths:
        print(row)
    print("-" * 36 + "\n")


# =================================================================================================
# 12. N-Queens Problem (Backtracking)
# =================================================================================================
Places N queens on an NxN board so none can attack each other.

In [8]:
def print_n_queens_board(board):
    for row in board:
        print(" ".join('Q' if col == 1 else '.' for col in row))

def is_safe_n_queens(board, row, col, N):
    # Check row on the left side
    for i in range(col):
        if board[row][i] == 1:
            return False
            
    # Check upper-left diagonal
    for i, j in zip(range(row, -1, -1), range(col, -1, -1)):
        if board[i][j] == 1:
            return False
            
    # Check lower-left diagonal
    for i, j in zip(range(row, N, 1), range(col, -1, -1)):
        if board[i][j] == 1:
            return False
            
    return True

def solve_n_queens_util(board, col, N):
    # Base case: All queens are placed
    if col >= N:
        return True
        
    # Try all rows in this column
    for r in range(N):
        if is_safe_n_queens(board, r, col, N):
            board[r][col] = 1 # Place queen
            
            # Recurse for the next column
            if solve_n_queens_util(board, col + 1, N):
                return True
                
            # Backtrack
            board[r][col] = 0 
            
    return False # No row worked in this column

def test_n_queens():
    print("--- 12. N-Queens Problem (Backtracking) ---")
    N = 4 # Size of the board (e.g., 4 for 4-Queens)
    print(f"Solving for {N}-Queens...")
    
    # Create an N x N board initialized to 0
    board = [[0 for _ in range(N)] for _ in range(N)]

    if solve_n_queens_util(board, 0, N):
        print(f"Solution found for {N}-Queens:")
        print_n_queens_board(board)
    else:
        print(f"No solution exists for {N}-Queens.")
    print("-" * 41 + "\n")

# =================================================================================================
# 13. Polynomial-Time Reduction (3-SAT to Vertex Cover)
# =================================================================================================
A demonstration of transforming one NP-complete problem into another.
This code *performs the reduction*, it doesn't solve the resulting problem.

In [9]:
def reduce_3sat_to_vc(clauses):
    """
    Reduces a 3-SAT formula to a Vertex Cover instance.
    'clauses' is a list of 3-tuples, e.g., [('x1', '!x2', 'x3')]
    """
    graph = {} # Adjacency list
    variables = set()
    num_clauses = len(clauses)
    
    def add_edge(u, v):
        graph.setdefault(u, []).append(v)
        graph.setdefault(v, []).append(u)

    # 1. Variable Gadgets
    for clause in clauses:
        for literal in clause:
            var = literal.replace('!', '')
            variables.add(var)
            
    for var in variables:
        add_edge(var, '!' + var)
        
    num_variables = len(variables)
    
    # 2. & 3. Clause Gadgets and Connections
    for i, (l1, l2, l3) in enumerate(clauses):
        c1 = f'c_{i},1'
        c2 = f'c_{i},2'
        c3 = f'c_{i},3'
        
        # Clause triangle
        add_edge(c1, c2)
        add_edge(c2, c3)
        add_edge(c3, c1)
        
        # Connection edges
        add_edge(c1, l1)
        add_edge(c2, l2)
        add_edge(c3, l3)

    # 4. Set k
    k = num_variables + 2 * num_clauses
    
    return graph, k

def test_3sat_to_vc_reduction():
    print("--- 13. 3-SAT to Vertex Cover Reduction ---")
    # Formula: (x1 OR x2 OR !x3) AND (!x1 OR x2 OR x3)
    formula = [('x1', 'x2', '!x3'), ('!x1', 'x2', 'x3')]
    
    graph, k = reduce_3sat_to_vc(formula)

    print(f"3-SAT Formula: {formula}")
    print(f"Resulting Vertex Cover 'k': {k}")
    print("Resulting Graph (partial view):")
    # Print first 5 nodes and their edges
    for i, node in enumerate(list(graph.keys())):
        if i >= 5: break
        print(f"  {node}: {graph[node]}")
    print("-" * 43 + "\n")

# =================================================================================================
# 14. Approximation Algorithm for Vertex Cover
# =================================================================================================
A 2-approximation algorithm for the NP-complete Vertex Cover problem.

In [10]:
def approx_vertex_cover(graph_adj):
    """
    Finds an approximate Vertex Cover for a graph.
    graph_adj is an adjacency list: {'A': ['B'], 'B': ['A', 'C', 'D'], ...}
    """
    vc = set()
    
    # 1. Convert adjacency list to an edge list to make it easier
    edges = set()
    for u in graph_adj:
        for v in graph_adj[u]:
            if (v, u) not in edges: # Avoid duplicate edges
                edges.add((u, v))
                
    # 2. Loop while edges remain
    while edges:
        # a. Pick any edge
        u, v = edges.pop()
        
        # b. Add both endpoints to the cover
        vc.add(u)
        vc.add(v)
        
        # c. Remove all edges connected to u or v
        # We build a new edge list, which is simpler
        edges_to_remove = set()
        for e_u, e_v in edges:
            if e_u == u or e_u == v or e_v == u or e_v == v:
                edges_to_remove.add((e_u, e_v))
        
        edges = edges - edges_to_remove
        
    return vc

def test_approx_vertex_cover():
    print("--- 14. Approximation Algorithm for Vertex Cover ---")
    graph = {
        'A': ['B'],
        'B': ['A', 'C', 'D'],
        'C': ['B', 'D'],
        'D': ['B', 'C']
    }
    # Optimal solution is {'B', 'C'} or {'B', 'D'} (size 2)

    vertex_cover = approx_vertex_cover(graph)
    print(f"Graph: {graph}")
    print(f"Approximate Vertex Cover: {vertex_cover}")
    print(f"Size: {len(vertex_cover)} (Note: Optimal is 2. This is <= 2 * Optimal)")
    print("-" * 52 + "\n")

# =================================================================================================
# Main execution block
# =================================================================================================

In [11]:
if __name__ == "__main__":
    test_mst_algorithms()
    test_dijkstra()
    # # test_sudoku_solver()
    # # test_floyd_warshall()
    # # test_n_queens()
    # test_3sat_to_vc_reduction()
    # test_approx_vertex_cover()

--- 8. Kruskal's and Prim's MST Algorithms ---
Graph Vertices: ['A', 'B', 'C', 'D']
Graph Edges: [(2, 'A', 'B'), (5, 'A', 'C'), (4, 'B', 'C'), (3, 'B', 'D'), (1, 'C', 'D')]

Kruskal's MST Edges (v1, v2, weight): [('C', 'D', 1), ('A', 'B', 2), ('B', 'D', 3)]
Kruskal's Total Cost: 6

Prim's MST Edges (v1, v2, weight): [('A', 'B', 2), ('B', 'D', 3), ('D', 'C', 1)]
Prim's Total Cost: 6
----------------------------------------

--- 9. Dijkstra's Algorithm ---
Graph: {'A': [(2, 'B'), (5, 'C')], 'B': [(2, 'A'), (4, 'C'), (3, 'D')], 'C': [(5, 'A'), (4, 'B'), (1, 'D')], 'D': [(3, 'B'), (1, 'C')]}
Shortest paths from source 'A': {'A': 0, 'B': 2, 'C': 5, 'D': 5}
-------------------------------

